<!-- config-banner -->
# DINO 1° — why 2× viscosity stabilises the forward and the adjoint, and what else does (2026-09-09)

**`viscRef` against `visc2x`, restarts from the 2× spin-up's year-170 and year-180 pickups**
(forward 2 yr and 10 yr, adjoint 30 d and 183 d), plus the kappa member M7's
own 5-yr adjoint 31046 restarted 183 d before its end. Runs and settings are
listed in `input/variants/stability_study/README.md` and
`input_tap/variants/stability_study/README.md`; this notebook reads their
monitor streams and `ADJ*` dumps and writes figures to
`DINO_1deg_outputs/analysis/stability_study/figures/` (several runs, so a
campaign directory, not a run directory).

Sections: 1 forward restarts (2 yr) — what the viscosity, its D/Z split, the
vorticity scheme, GM and the grid-Reynolds floor do to the flow; 2 the 10-yr
continuations; 3 adjoint growth over 30 d for every switch tried, GM included;
4 the M7 half-year adjoint: the original blow-up reproduced, and what scheme
30 and the Reynolds floor do to it; 5 conclusions.

In [ ]:
import os, glob, re, collections, filecmp, warnings
import numpy as np
import matplotlib.pyplot as plt
import xmitgcm
warnings.filterwarnings("ignore")
RUNS = "/scratch2/tshahriar/DINO_1deg_outputs/runs"
FIG = "/scratch2/tshahriar/DINO_1deg_outputs/analysis/stability_study/figures"; os.makedirs(FIG, exist_ok=True)
SPIN = RUNS + "/forward/spinup_200yr_visc2x/DINO_1deg_frd_200yr_from_rest_visc2x_run30983"
IT170, IT180, SPD, SPY = 2986560, 3162240, 48, 17568

def find(pattern):
    hits = sorted(glob.glob(f"{RUNS}/{pattern}") + glob.glob(f"{RUNS}/*/{pattern}") + glob.glob(f"{RUNS}/*/*/{pattern}"))
    assert hits, pattern
    return hits[-1]

def mon(stdout):
    b, cur = collections.OrderedDict(), None
    for l in open(stdout):
        if "%MON" not in l: continue
        s = l.split("%MON", 1)[1].strip()
        if s.startswith("time_tsnumber"):
            cur = int(s.split("=")[1]); b[cur] = {}
        elif cur is not None and "=" in s:
            k, v = s.split("=", 1)
            try: b[cur][k.strip()] = float(v)
            except ValueError: pass
    return b

def series(b, key, it0=IT170, lo=None, hi=None):
    its = np.array([i for i in b if (lo is None or i >= lo) and (hi is None or i <= hi)])
    return (its - it0) / SPY, np.array([b[i].get(key, np.nan) for i in its])

def speed_series(b, **kw):
    yr, umax = series(b, "dynstat_uvel_max", **kw); _, umin = series(b, "dynstat_uvel_min", **kw)
    _, vmax = series(b, "dynstat_vvel_max", **kw); _, vmin = series(b, "dynstat_vvel_min", **kw)
    _, ke = series(b, "ke_max", **kw); _, kem = series(b, "ke_mean", **kw)
    return yr, np.maximum(umax, -umin), np.maximum(vmax, -vmin), ke, kem

## 1. Forward restarts, 2 yr from year 170

Peak speeds and kinetic energy from the monitor stream (every 5 d), each run
against the spin-up's own years 170–172 (monthly monitor). The tags are the
`stability_study` variants; `viscRef` is DINO's reference viscosity on both the
divergence (D) and vorticity (Z) parts, `viscDref_Z2x` doubles only Z.

In [ ]:
tags = ["viscRef", "viscRef_vort3", "viscRef_vort2", "viscRef_cori1", "viscRef_A4Grid1p0e-2",
        "viscRef_gmOff", "visc2x_gmOff", "viscD2x_Zref", "viscDref_Z2x", "viscRef_ReMax2"]
fwd = {t: mon(find(f"DINO_1deg_frd_2yr_from170yrPk_{t}_run*") + "/STDOUT.0000") for t in tags}
spin = mon(SPIN + "/STDOUT.0000")
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5), constrained_layout=True)
yr, u, v, ke, kem = speed_series(spin, lo=IT170, hi=IT170 + 2 * SPY)
for a, s, lab in zip(ax, (u, v, ke), ("max |u| [m/s]", "max |v| [m/s]", "ke_max [m²/s²]")):
    a.plot(yr, s, "k", lw=3, alpha=0.4, label="spin-up (2×, GM on)"); a.set_ylabel(lab); a.set_xlabel("years after 170"); a.grid(alpha=0.4)
print(f"{'tag':22s} {'|u|max':>7s} {'|v|max':>7s} {'ke_max':>7s} {'ke_mean':>8s}   (values at 2 yr)")
print(f"{'spin-up 2x':22s} {u[-1]:7.2f} {v[-1]:7.2f} {ke[-1]:7.2f} {kem[-1]*1e4:8.2f}e-4")
for t, b in fwd.items():
    yr, u, v, ke, kem = speed_series(b)
    ls = "--" if t in ("viscDref_Z2x", "viscRef_ReMax2", "visc2x_gmOff") else "-"
    for a, s in zip(ax, (u, v, ke)): a.plot(yr, s, ls, lw=1.5, label=t)
    print(f"{t:22s} {u[-1]:7.2f} {v[-1]:7.2f} {ke[-1]:7.2f} {kem[-1]*1e4:8.2f}e-4")
ax[2].legend(fontsize=8, ncol=2)
fig.savefig(FIG + "/forward_2yr_restarts_speeds.png", dpi=120); plt.show()

### Where `viscAhReMax=2.` adds viscosity

`viscAhReMax` is a floor, A ≥ |u|·L/Re_max, so it leaves DINO's field alone
where the flow is slower than 0.27 m/s (Re_Δ = 2 at that speed for DINO's
A = 0.27·Δx/2) and raises it only in the jets. The last monthly `VISCAHZ` of
the 2-yr run against the reference field 0.135·dxC.

In [ ]:
r = find("DINO_1deg_frd_2yr_from170yrPk_viscRef_ReMax2_run*")
its = sorted(int(p.split(".")[-2]) for p in glob.glob(r + "/viscDiag.*.meta"))
ds = xmitgcm.open_mdsdataset(r, grid_dir=r, prefix=["viscDiag"], geometry="curvilinear", iters=[its[-1]], delta_t=1800.)
ratio = (ds.VISCAHZ.isel(time=0) / (0.135 * ds.dxC)).where(ds.hFacC > 0)
wet = (ds.hFacC > 0).values; rv = ratio.values
print(f"VISCAHZ / reference: max {np.nanmax(rv):.2f}; wet fraction > 1.05: {(rv[wet] > 1.05).mean():.4f}, > 1.5: {(rv[wet] > 1.5).mean():.4f}, > 2: {(rv[wet] > 2).mean():.4f}")
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
for a, k in zip(ax, (0, 5)):
    im = a.pcolormesh(ds.XC, ds.YC, rv[k], vmin=1, vmax=2.5, cmap="magma_r"); fig.colorbar(im, ax=a, label="VISCAHZ / (0.135 dxC)")
    a.set_title(f"viscRef_ReMax2, k={k} (z={float(ds.Z[k]):.0f} m), last month of year 2")
fig.savefig(FIG + "/ReMax2_viscosity_ratio_maps.png", dpi=120); plt.show()

## 2. The 10-yr continuations

`viscRef` and `viscRef_ReMax2` for 10 yr from year 170, monthly monitor,
against the spin-up's years 170–180.

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(20, 4.2), constrained_layout=True)
yr, u, v, ke, kem = speed_series(spin, lo=IT170, hi=IT170 + 10 * SPY)
_, th = series(spin, "dynstat_theta_min", lo=IT170, hi=IT170 + 10 * SPY)
for a, s, lab in zip(ax, (u, v, ke, th), ("max |u| [m/s]", "max |v| [m/s]", "ke_max", "theta_min [°C]")):
    a.plot(yr, s, "k", lw=3, alpha=0.4, label="spin-up (2×, GM on)"); a.set_ylabel(lab); a.set_xlabel("years after 170"); a.grid(alpha=0.4)
for t in ("viscRef", "viscRef_ReMax2"):
    g = glob.glob(f"{RUNS}/forward/DINO_1deg_frd_10yr_from170yrPk_{t}_run*") + glob.glob(f"{RUNS}/forward/*/DINO_1deg_frd_10yr_from170yrPk_{t}_run*")
    if not g: print(t, "10-yr run not found"); continue
    b = mon(g[0] + "/STDOUT.0000"); yr, u, v, ke, kem = speed_series(b); _, th = series(b, "dynstat_theta_min")
    ended = "Run ended" in open(g[0] + "/run_timing.txt").read()
    print(f"{t}: {len(yr)} monitor blocks, last year {yr[-1]:.2f}, ended={ended}; max |u| {u.max():.2f}, max |v| {v.max():.2f}, ke_max {ke.max():.2f}")
    for a, s in zip(ax, (u, v, ke, th)): a.plot(yr, s, lw=1.5, label=t)
ax[0].legend(fontsize=9)
fig.savefig(FIG + "/forward_10yr_continuations.png", dpi=120); plt.show()

## 3. Adjoint growth over 30 d, from the 180-yr pickup

Wet-RMS of `ADJtheta` per 5-d dump against lead time, one curve per switch.
There is no adjoint monitor stream in these builds, so the dumps are the
record. The reference adjoint (2×, GM off) is 31140.

In [ ]:
def adj_curve(d, var="ADJtheta"):
    fs = sorted(glob.glob(f"{d}/{var}.*.data"))
    its = np.array([int(os.path.basename(f).split(".")[1]) for f in fs]); lead = (its.max() - its) / SPD
    rms = np.array([np.sqrt(np.mean(np.fromfile(f, dtype=">f4").astype("f8") ** 2)) for f in fs])
    return lead, rms
adj30 = collections.OrderedDict()
adj30["visc2x, GM off (31140)"] = find("DINO_1deg_tapAdj_nocheckpoint_30d_from180yrPk_visc2x_run31140")
for tag in ("viscRef", "viscRef_vort3", "viscRef_ReMax2", "visc2x_gmOn", "viscRef_gmOn", "visc2x_gmFwd", "viscRef_gmFwd", "visc2x_approxAdv", "viscRef_approxAdv"):
    g = glob.glob(f"{RUNS}/adjoint/DINO_1deg_tapAdj_*_30d_from180yrPk_{tag}_run*") + glob.glob(f"{RUNS}/adjoint/*/DINO_1deg_tapAdj_*_30d_from180yrPk_{tag}_run*")
    if g: adj30[tag] = sorted(g)[-1]
fig, ax = plt.subplots(figsize=(9, 5))
for lab, d in adj30.items():
    lead, rms = adj_curve(d)
    ax.semilogy(lead, rms, "o-", label=f"{lab} ({os.path.basename(d).split('_run')[-1]})")
    print(f"{lab:26s} rms(ADJtheta) at lead 5/15/25 d: " + "  ".join(f"{r:.2e}" for l, r in zip(lead, rms) if int(l) in (5, 15, 25)))
ax.set_xlabel("lead [d]"); ax.set_ylabel("wet RMS of ADJtheta"); ax.grid(alpha=0.4); ax.legend(fontsize=8)
ax.set_title("30-d adjoints from the 180-yr pickup")
fig.savefig(FIG + "/adjoint_30d_growth_switches.png", dpi=120); plt.show()

## 4. The M7 half-year adjoint: the blow-up reproduced, and two candidate cures

Kappa member M7 (32× κ_v, 2× viscosity, GM off) blew up 132 d before the end of
its 5-yr adjoint 31046. Its namelist restarted from the monthly pickup that run
wrote 183 d before its end integrates the same trajectory and the same
terminal-30-d cost, so its 183-d adjoint is the last 183 d of 31046 — checked
byte for byte below for the control — and the same run with scheme 30 (DST3
without the flux limiter, both sweeps) or with `viscAhReMax=2.` shows what each
does to the blow-up.

In [ ]:
ref46 = find("DINO_1deg_tapAdj_ckpAll_5yr_M7_run31046")
m7 = collections.OrderedDict()
for tag, lab in (("M7_lastHalfYr", "control (scheme 33, 2×)"), ("M7_lastHalfYr_adv30", "scheme 30 both sweeps"), ("M7_lastHalfYr_ReMax2", "+ viscAhReMax=2")):
    g = glob.glob(f"{RUNS}/adjoint/DINO_1deg_tapAdj_ckpAll_183d_{tag}_run*") + glob.glob(f"{RUNS}/adjoint/*/DINO_1deg_tapAdj_ckpAll_183d_{tag}_run*")
    if g: m7[lab] = sorted(g)[-1]
lead46, rms46 = adj_curve(ref46); sel = lead46 <= 183
fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(lead46[sel], rms46[sel], "k", lw=3, alpha=0.4, label="M7 5-yr adjoint 31046 (last 183 d)")
for lab, d in m7.items():
    lead, rms = adj_curve(d); ax.semilogy(lead, rms, "o-", ms=3, label=f"{lab} ({os.path.basename(d).split('_run')[-1]})")
    same = sum(filecmp.cmp(f, ref46 + "/" + os.path.basename(f), shallow=False) for f in glob.glob(d + "/ADJtheta.*.data") if os.path.exists(ref46 + "/" + os.path.basename(f)))
    fin = all(np.isfinite(np.fromfile(f, dtype=">f4")).all() for f in glob.glob(d + "/ADJtheta.*.data"))
    print(f"{lab:26s}: {len(lead)} dumps, {same} byte-identical to 31046, all finite: {fin}; rms at lead 30/90/150/180 d: " + "  ".join(f"{r:.2e}" for l, r in zip(lead, rms) if int(round(l)) in (30, 90, 150, 180)))
ax.set_xlabel("lead [d]"); ax.set_ylabel("wet RMS of ADJtheta"); ax.grid(alpha=0.4); ax.legend(fontsize=9)
ax.set_title("M7: the last 183 d of the 5-yr adjoint, restarted")
fig.savefig(FIG + "/M7_halfyear_adjoint_blowup_and_cures.png", dpi=120); plt.show()

## 5. Conclusions

(filled in from the runs above; see the stability_study READMEs and the DINO
`TODO.md` entry of 2026-09-09 for the numbers)